### bronze - fhv (small bases)
using schema inference here instead of an explicit schema like the other 3.
FHV is small and sparse, and the column naming has shifted around more across
years than yellow/green/fhvhv - wasn't confident enough hardcoding exact names
for months I haven't actually inspected. easier to let spark figure it out and
grab the pickup col dynamically below.

also still missing april 2026 for this one, TLC hadn't published it yet as of
the download (small bases get filing extensions sometimes). not blocking on it.

In [0]:
from pyspark.sql.functions import year, month, current_timestamp, col

In [0]:
catalog = "nyc_taxi"
raw_path = f"/Volumes/{catalog}/raw/trip_data/fhv"
bronze_table = f"{catalog}.bronze.fhv"

In [0]:
raw_df = (
    spark.read
    .format("parquet")
    .option("mergeSchema", "true")
    .load(raw_path)
)

raw_df.printSchema()
print(raw_df.count())

root
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropOff_datetime: timestamp_ntz (nullable = true)
 |-- PUlocationID: long (nullable = true)
 |-- DOlocationID: long (nullable = true)
 |-- SR_Flag: long (nullable = true)
 |-- Affiliated_base_number: string (nullable = true)

23939185


In [0]:
# pickup column name has moved around across TLC's schema revisions for this
# dataset specifically, so find it instead of hardcoding

candidates = [c for c in raw_df.columns if "pickup" in c.lower() and "datetime" in c.lower()]
if not candidates:
    raise ValueError(f"no pickup datetime column found, columns are: {raw_df.columns}")
pickup_col = candidates[0]
print("using", pickup_col)

using pickup_datetime


In [0]:
bronze_df = (
    raw_df
    .withColumn("pickup_year", year(col(pickup_col)))
    .withColumn("pickup_month", month(col(pickup_col)))
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
(
    bronze_df.write
    .format("delta")
    .partitionBy("pickup_year", "pickup_month")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

print(f"wrote {bronze_df.count()} rows -> {bronze_table}")

wrote 23939185 rows -> nyc_taxi.bronze.fhv


In [0]:
display(spark.sql(f"""
    select pickup_year, pickup_month, count(*) as trip_count
    from {bronze_table}
    group by pickup_year, pickup_month
    order by pickup_year, pickup_month
"""))

pickup_year,pickup_month,trip_count
2025,5,2210721
2025,6,2231731
2025,7,2187536
2025,8,2256854
2025,9,2149292
2025,10,2446615
2025,11,2278604
2025,12,1926891
2026,1,1941722
2026,2,1948529
